In [13]:
mask = (1 << 14) - 1
aa = (0xa7e0 >> 2) & mask
print(hex(aa))
print(bin(aa))

0x29f8
0b10100111111000


In [7]:

def bitrev(x, logn):
    r = 0
    for i in range(logn):
        r = (r << 1) | (x & 1)
        x >>= 1
    return r

print(hex(bitrev(0x7, 8)))

0xe0


In [20]:
temp = (bitrev(0xe0, 8) << 8) | bitrev(0xa7, 8)
print(bin(temp))
temp1 = temp & mask
temp2 = bitrev(temp1, 16) >> 2
print(hex(temp2))

0b11111100101
0x29f8


In [ ]:
def poly_modq_decode(data: bytes, n: int = 512, q: int = 12289):
    """
    Python equivalent of Falcon-style poly_modq_decode().

    Args:
        data: input byte stream
        n: number of coefficients (N)
        q: modulus (default 12289)

    Returns:
        (coeffs, in_len) on success
        (None, 0) on failure
    """
    in_len = ((n * 14) + 7) >> 3

    acc = 0
    acc_len = 0
    u = 0
    coeffs = [0] * n
    idx = 0

    while u < n:
        if (idx >= len(data)):
            return coeffs, in_len
        
        acc = ((acc << 8) | data[idx]) & 0xFFFFFFFF
        idx += 1
        acc_len += 8

        if acc_len >= 14:
            acc_len -= 14
            w = (acc >> acc_len) & 0x3FFF
            if w >= q:
                print(hex(w))
                return None, 0
            coeffs[u] = w
            u += 1

    # Unused tail bits must be zero

    return coeffs, in_len

In [100]:
def poly_modq_decode_ours(data: bytes, n: int = 512, q: int = 12289):
    acc = 0
    acc_len = 0
    u = 0
    coeffs = [0] * n
    idx = 0

    while u < n: 
        if (idx >= len(data)):
            return coeffs
               
        acc = acc | ((bitrev(data[idx], 8) << acc_len))
        idx += 1
        acc_len += 8

        if acc_len >= 14:
            acc_len -= 14
            w = acc & 0x3FFF
            w = bitrev(w, 16) >> 2
            acc >>= 14
            if w >= q:
                return None, 0
            coeffs[u] = w
            u += 1

    # Unused tail bits must be zero

    return coeffs

def poly_modq_encode_ours(coeffs, q: int = 12289) -> bytes:
    acc = 0
    acc_len = 0
    out = bytearray()

    for w in coeffs:
        if w < 0 or w >= q:
            raise ValueError(f"coefficient out of range: {w}")

        # inverse of decode_ours step: w = bitrev(raw14,16)>>2
        # so raw14 = low 14 bits of (bitrev(w,14))
        raw14 = bitrev(w, 14)

        acc |= (raw14 << acc_len)
        acc_len += 14

        while acc_len >= 8:
            b = acc & 0xFF
            out.append(bitrev(b, 8))
            acc >>= 8
            acc_len -= 8

    # flush remaining bits (if any), must be padded with zeros
    if acc_len > 0:
        b = acc & 0xFF
        out.append(bitrev(b, 8))

    return bytes(out)

In [101]:
data = bytes.fromhex('a5e07c01eb60512ece67e7886512bc0a')
data

b'\xa5\xe0|\x01\xeb`Q.\xceg\xe7\x88e\x12\xbc\n'

In [98]:
arr = poly_modq_decode(data)
print([hex(x) for x in arr[0]])

['0x2978', '0x7c0', '0x7ad', '0x2051', '0xbb3', '0x267e', '0x1e21', '0x2512', '0x2f02', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0

In [99]:
arr = poly_modq_decode_ours(data)
print([hex(x) for x in arr])

['0x2978', '0x7c0', '0x7ad', '0x2051', '0xbb3', '0x267e', '0x1e21', '0x2512', '0x2f02', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0x0', '0

In [104]:
data2 = poly_modq_encode_ours(arr)
assert data[:10] == data2[:10]